# Topological Observables on Spiking Networks

**SC-NeuroCore v3.14** — Differential geometry meets neuromorphic computing.

SC-NeuroCore is the first SNN framework to expose topological
observables from differential geometry on coupled oscillator
networks. These measure structural properties of synchronisation
that are invisible to standard spike statistics.

1. **Winding number** — topological invariant: how many times the phase wraps $S^1$
2. **Ollivier-Ricci curvature** — positive = community, negative = bottleneck
3. **Sheaf consistency defect** — obstruction to global coherence
4. **Connection curvature** — parallel transport obstruction between layers

These map directly to consciousness observables in the SCPN model:
sheaf defect $\approx 1 - R_{\text{Kuramoto}}$, curvature identifies
integration bottlenecks.

> © 1998–2026 Miroslav Šotek. All rights reserved.  
> License: GNU AFFERO GENERAL PUBLIC LICENSE v3 | Commercial Licensing Available  
> Contact: www.anulum.li | protoscience@anulum.li

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sc_neurocore.math.topology import (
    winding_number,
    ollivier_ricci_curvature,
    sheaf_consistency_defect,
    connection_curvature,
)

print("SC-NeuroCore topological observables demo")

## 1. Winding Number

The winding number counts how many times a phase trajectory
wraps around $S^1 = [0, 2\pi)$. It is a topological invariant:
continuous deformations cannot change it.

For a single oscillator with constant frequency $\omega$,
the winding number over time $T$ is $\lfloor \omega T / 2\pi \rfloor$.

In [ ]:
# Single oscillator: 3 full rotations
T = 1000
omega = 3 * 2 * np.pi / T  # 3 wraps in T steps
phases_3wrap = np.array([(omega * t) % (2 * np.pi) for t in range(T)])
w3 = winding_number(phases_3wrap)

# 0 wraps (subthreshold oscillation)
phases_0wrap = np.linspace(0, np.pi, T)  # half rotation, never wraps
w0 = winding_number(phases_0wrap)

# 10 wraps
omega10 = 10 * 2 * np.pi / T
phases_10wrap = np.array([(omega10 * t) % (2 * np.pi) for t in range(T)])
w10 = winding_number(phases_10wrap)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, phases, w, label in [
    (axes[0], phases_0wrap, w0, f"W = {w0} (subthreshold)"),
    (axes[1], phases_3wrap, w3, f"W = {w3} (3 wraps)"),
    (axes[2], phases_10wrap, w10, f"W = {w10} (10 wraps)"),
]:
    ax.plot(phases, linewidth=0.5)
    ax.set_xlabel("Time step")
    ax.set_ylabel("Phase (rad)")
    ax.set_title(label)
    ax.set_ylim(-0.5, 2 * np.pi + 0.5)

plt.tight_layout()
plt.show()

print(f"Winding numbers: 0-wrap={w0}, 3-wrap={w3}, 10-wrap={w10}")

## 2. Ollivier-Ricci Curvature

Ricci curvature on graphs (Ollivier 2009) measures whether
neighbourhoods of connected nodes converge or diverge:

$$\kappa(i,j) = 1 - \frac{W_1(\mu_i, \mu_j)}{d(i,j)}$$

- **Positive** curvature: nodes share many neighbours (community)
- **Negative** curvature: nodes bridge separate clusters (bottleneck)
- **Zero**: tree-like or lattice-like structure

In [ ]:
N = 16

# Complete graph: all nodes connected → high positive curvature
K_complete = np.ones((N, N)) * 0.5
np.fill_diagonal(K_complete, 0)

# Ring graph: each node connected to 2 neighbours → low curvature
K_ring = np.zeros((N, N))
for i in range(N):
    K_ring[i, (i + 1) % N] = 0.5
    K_ring[i, (i - 1) % N] = 0.5

# Two clusters with a single bridge → bridge has negative curvature
K_bridge = np.zeros((N, N))
half = N // 2
for i in range(half):
    for j in range(i + 1, half):
        K_bridge[i, j] = K_bridge[j, i] = 0.3
for i in range(half, N):
    for j in range(i + 1, N):
        K_bridge[i, j] = K_bridge[j, i] = 0.3
K_bridge[half - 1, half] = K_bridge[half, half - 1] = 0.3  # bridge

graphs = {
    "Complete": K_complete,
    "Ring": K_ring,
    "Two-cluster bridge": K_bridge,
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, K) in zip(axes, graphs.items()):
    curvatures = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if K[i, j] > 0:
                curvatures[i, j] = ollivier_ricci_curvature(K, i, j)
    im = ax.imshow(curvatures, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_title(f"{name}\nmean κ = {curvatures[K > 0].mean():.3f}")
    ax.set_xlabel("Node j")
    ax.set_ylabel("Node i")

plt.colorbar(im, ax=axes[-1], label="Ricci curvature κ")
plt.tight_layout()
plt.show()

## 3. Sheaf Consistency Defect

In sheaf theory, a global section exists iff local sections
agree on overlaps. For SCPN, the coupling matrix defines
overlaps, and phases define local sections.

$$\text{defect} = \frac{1}{N^2} \sum_{i,j} |K_{ij}| \cdot |1 - \cos(\theta_i - \theta_j)|$$

- **Synchronised** ($\theta_i = \theta_j$): defect = 0
- **Incoherent**: defect > 0, proportional to $1 - R_{\text{Kuramoto}}$

In [ ]:
# Sweep from synchronised to random phases
K_test = K_complete[:8, :8]  # 8-node complete graph
n_test = 8

noise_levels = np.linspace(0, 2 * np.pi, 30)
defects = []
kuramoto_R = []

for noise in noise_levels:
    phases = np.zeros(n_test) + noise * np.random.default_rng(42).uniform(0, 1, n_test)
    phases = phases % (2 * np.pi)
    defects.append(sheaf_consistency_defect(phases, K_test))
    # Kuramoto order parameter for comparison
    R = abs(np.mean(np.exp(1j * phases)))
    kuramoto_R.append(R)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(noise_levels, defects, "o-", markersize=3)
axes[0].set_xlabel("Phase noise amplitude (rad)")
axes[0].set_ylabel("Sheaf consistency defect")
axes[0].set_title("Sheaf defect vs phase noise")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(kuramoto_R, defects, s=15, alpha=0.7)
axes[1].set_xlabel("Kuramoto R (order parameter)")
axes[1].set_ylabel("Sheaf defect")
axes[1].set_title("Sheaf defect ≈ 1 − R (coupling-weighted)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Connection Curvature (PGBO)

The connection curvature $F_{ij} = K_{ij} \cos(\theta_i - \theta_j)$
measures the obstruction to parallel transport between nodes.
It is the discrete analogue of the Riemann curvature tensor
for the phase bundle over the coupling graph.

In [ ]:
# Synchronised case: F_ij = K_ij (no obstruction)
phases_sync = np.zeros(n_test)
F_sync = connection_curvature(phases_sync, K_test)

# Anti-phase: alternating 0 and pi
phases_anti = np.array([0, np.pi] * (n_test // 2))
F_anti = connection_curvature(phases_anti, K_test)

# Random phases
phases_rand = np.random.default_rng(42).uniform(0, 2 * np.pi, n_test)
F_rand = connection_curvature(phases_rand, K_test)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, F, title in [
    (axes[0], F_sync, f"Synchronised\nmean F = {F_sync[K_test > 0].mean():.3f}"),
    (axes[1], F_anti, f"Anti-phase\nmean F = {F_anti[K_test > 0].mean():.3f}"),
    (axes[2], F_rand, f"Random\nmean F = {F_rand[K_test > 0].mean():.3f}"),
]:
    im = ax.imshow(F, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
    ax.set_title(title)

plt.colorbar(im, ax=axes[-1], label="Connection curvature F")
plt.tight_layout()
plt.show()

## Summary

| Observable | Formula | Interpretation |
|-----------|---------|----------------|
| Winding number | $W = \lfloor \sum \Delta\theta / 2\pi \rfloor$ | Phase wrapping count (topological invariant) |
| Ricci curvature | $\kappa = 1 - W_1(\mu_i, \mu_j)/d$ | Community (+) vs bottleneck (−) |
| Sheaf defect | $\sum|K||1-\cos\Delta\theta|/N^2$ | Obstruction to global coherence |
| Connection curvature | $F_{ij} = K_{ij}\cos\Delta\theta$ | Parallel transport obstruction |

These observables map to consciousness metrics in the SCPN model:
- Sheaf defect → complementary to integrated information ($\Phi$)
- Ricci curvature → identifies integration bottlenecks (critical links)
- Winding number → topological protection of phase-encoded memory

No other SNN framework exposes sheaf cohomology, Ricci curvature,
or connection forms on spiking networks.